In [14]:
# ==== Config ====
BASE_MODEL = "Qwen/Qwen3-4B"

HF_ID = "gtfintechlab/financial_phrasebank_sentences_allagree"
HF_CONFIG = "5768"   # required

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
LABELS = ["negative", "neutral", "positive"]

# sweep settings
Ns = [16, 32, 64]
SEEDS = [0, 1, 2]  # start with [0], then change to [0,1,2]

# training/eval
MAX_LENGTH = 512
MAX_NEW_TOKENS = 2
MAX_STEPS = 200
TRAIN_BSZ = 2
GRAD_ACCUM = 8
LR = 2e-3
EVAL_MAX = 300

# P-Tuning v2
NUM_VIRTUAL_TOKENS = 20
ENCODER_HIDDEN_SIZE = 128
ENCODER_NUM_LAYERS = 2

RESULTS_CSV = "results.csv"

In [2]:
!pip -q install -U transformers datasets accelerate peft scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 155.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.0 MB/s eta 0:00:00


In [3]:
import os, gc, csv, time, random, datetime
from collections import defaultdict

import numpy as np
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer,
)
from peft import PromptEncoderConfig, get_peft_model, TaskType

In [6]:
from huggingface_hub import login
login()

In [15]:
ds = load_dataset(HF_ID, HF_CONFIG)

def to_examples(split):
    out = []
    for i, row in enumerate(ds[split]):
        y = LABEL_MAP[int(row["label"])]
        out.append({"id": f"{split}_{i}", "x": row["sentence"], "y": y, "label": y})
    return out

all_train = to_examples("train")
eval_set  = to_examples("test")  # fixed eval split (good)
print("train:", len(all_train), "test:", len(eval_set), "example:", all_train[0])

train: 1584 test: 680 example: {'id': 'train_0', 'x': "The equipment will be made at Vaahto 's plant in Hollola in Finland , and delivery is scheduled for the first quarter of 2009 .", 'y': 'neutral', 'label': 'neutral'}


In [8]:
def make_ordered(train_examples, seed):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for ex in train_examples:
        by_label[ex["label"]].append(ex)
    for lab in by_label:
        rng.shuffle(by_label[lab])

    ordered = []
    labels_sorted = sorted(by_label.keys())
    i = 0
    while True:
        progressed = False
        for lab in labels_sorted:
            if i < len(by_label[lab]):
                ordered.append(by_label[lab][i])
                progressed = True
        if not progressed:
            break
        i += 1
    return ordered

def train_text(ex):
    return (
        "Task: Financial sentiment classification.\n"
        "Output EXACTLY one label: negative, neutral, or positive.\n\n"
        f"Input:\n{ex['x']}\n"
        f"Label: {ex['y']}"
    )

def eval_prompt(x):
    return (
        "Task: Financial sentiment classification.\n"
        "Output EXACTLY one label: negative, neutral, or positive.\n\n"
        f"Input:\n{x}\n"
        "Label:"
    )

def parse_label(text):
    t = (text or "").strip().lower()
    for lab in LABELS:
        if lab in t.split():
            return lab
    for lab in LABELS:
        if lab in t:
            return lab
    return "unknown"

In [9]:
def build_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    peft_config = PromptEncoderConfig(
        peft_type="P_TUNING",
        task_type=TaskType.CAUSAL_LM,
        num_virtual_tokens=NUM_VIRTUAL_TOKENS,
        token_dim=model.config.hidden_size,
        num_layers=model.config.num_hidden_layers,
        num_attention_heads=model.config.num_attention_heads,
        encoder_hidden_size=ENCODER_HIDDEN_SIZE,
        encoder_num_layers=ENCODER_NUM_LAYERS,
    )
    pt_model = get_peft_model(model, peft_config)
    return pt_model, tokenizer

In [13]:
def train_one_run(pt_model, tokenizer, train_N, output_dir):
    ds_train = Dataset.from_dict({"text": [train_text(ex) for ex in train_N]})

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

    ds_tok = ds_train.map(tok, batched=True, remove_columns=["text"])
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=TRAIN_BSZ,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        fp16=True,
        logging_steps=10,
        save_steps=MAX_STEPS,
        report_to=[],
    )

    trainer = Trainer(model=pt_model, args=args, train_dataset=ds_tok, data_collator=collator)
    trainer.train()
    pt_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

In [10]:
def gpu_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def gen_one(pt_model, tokenizer, x):
    inputs = tokenizer(
        eval_prompt(x),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(pt_model.device)

    prompt_len = inputs["input_ids"].shape[-1]

    gpu_sync()
    t0 = time.perf_counter()
    out = pt_model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gpu_sync()
    t1 = time.perf_counter()

    gen_ids = out[0][prompt_len:]
    txt = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return txt, int(gen_ids.shape[-1]), (t1 - t0)

def evaluate(pt_model, tokenizer, max_eval=EVAL_MAX):
    pt_model.eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    lat = []
    total_new = 0
    total_t = 0.0
    correct = 0
    unk = 0

    for ex in eval_set[:max_eval]:
        txt, new_toks, dt = gen_one(pt_model, tokenizer, ex["x"])
        pred = parse_label(txt)

        if pred == "unknown":
            unk += 1
        if pred == ex["y"]:
            correct += 1

        lat.append(dt * 1000)
        total_new += new_toks
        total_t += dt

    peak = (torch.cuda.max_memory_allocated()/(1024**2)) if torch.cuda.is_available() else None
    n_eval = min(max_eval, len(eval_set))
    return {
        "accuracy": correct / n_eval,
        "unknown_frac": unk / n_eval,
        "lat_mean_ms": float(np.mean(lat)),
        "lat_p50_ms": float(np.percentile(lat, 50)),
        "lat_p95_ms": float(np.percentile(lat, 95)),
        "throughput_new_toks_per_s": (total_new / total_t) if total_t > 0 else None,
        "peak_vram_mb": peak,
        "eval_n": n_eval,
    }

In [11]:
def append_csv(row, csv_path=RESULTS_CSV):
    write_header = not os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if write_header:
            w.writeheader()
        w.writerow(row)

In [16]:
for seed in SEEDS:
    ordered = make_ordered(all_train, seed)

    for N in Ns:
        out_dir = f"ptuning_fpb_{BASE_MODEL.split('/')[-1]}_{HF_CONFIG}_N{N}_seed{seed}"
        print(f"\n==== seed={seed} N={N} ====")

        # Fresh model each run (CRITICAL)
        pt_model, tokenizer = build_model_and_tokenizer()

        train_N = ordered[:N]
        train_one_run(pt_model, tokenizer, train_N, out_dir)

        metrics = evaluate(pt_model, tokenizer)
        print(metrics)

        row = {
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "base_model": BASE_MODEL,
            "hf_id": HF_ID,
            "hf_config": HF_CONFIG,
            "N": N,
            "seed": seed,
            "max_length": MAX_LENGTH,
            "max_new_tokens": MAX_NEW_TOKENS,
            "max_steps": MAX_STEPS,
            "train_bsz": TRAIN_BSZ,
            "grad_accum": GRAD_ACCUM,
            "lr": LR,
            "num_virtual_tokens": NUM_VIRTUAL_TOKENS,
            "encoder_hidden_size": ENCODER_HIDDEN_SIZE,
            "encoder_num_layers": ENCODER_NUM_LAYERS,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            **metrics,
        }
        append_csv(row)

        # free VRAM
        del pt_model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nDone. Wrote:", RESULTS_CSV)


==== seed=0 N=16 ====


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss
10,3.721869
20,3.229290
30,2.563867
40,1.792736
50,1.549632
60,1.429144
70,1.342163
80,1.220752
90,1.157712
100,0.988654


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


...

{'accuracy': 0.8333333333333334, 'unknown_frac': 0.0033333333333333335, 'lat_mean_ms': 116.67513962001976, 'lat_p50_ms': 116.03406849985731, 'lat_p95_ms': 121.62223120008095, 'throughput_new_toks_per_s': 17.141612227878827, 'peak_vram_mb': 7727.783203125, 'eval_n': 300}

==== seed=0 N=64 ====


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Step,Training Loss
10,3.510142
20,2.825727
30,1.951260
40,1.734163


KeyboardInterrupt: 